# Notion Database Integration

Track H5N1 antibody design pipeline progress in Notion

## 0. Setup

In [ ]:
from google.colab import drive, userdata
import os, subprocess, json
from pathlib import Path
import pandas as pd

print('[STEP 1] Mount & setup')
drive.mount('/content/drive')

repo_path = Path('/content/h5n1')
if not repo_path.exists():
    subprocess.run('git clone https://github.com/Dajeong0315/h5n1-antibody-design.git /content/h5n1', shell=True, check=True)

os.chdir('/content/h5n1')
subprocess.run('git pull', shell=True, check=True)

try:
    NOTION_TOKEN = userdata.get('NOTION_TOKEN')
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    GITHUB_USER = userdata.get('GITHUB_USER')
    print('[OK] Credentials loaded')
except:
    NOTION_TOKEN = GITHUB_TOKEN = GITHUB_USER = ''
    print('[WARN] Missing credentials')

!pip install -q notion-client pandas
print('[OK] Setup complete')

## 1. Create/Connect to Notion Database

In [ ]:
from notion_client import Client
from datetime import datetime

if not NOTION_TOKEN:
    print('[ERROR] NOTION_TOKEN not set!')
    print('[TIP] Set it in Colab Secrets')
else:
    notion = Client(auth=NOTION_TOKEN)
    
    # List existing databases
    try:
        response = notion.search(query='H5N1', filter={'value': 'database', 'property': 'object'})
        if response['results']:
            db_id = response['results'][0]['id']
            print(f'[OK] Found existing database: {response["results"][0]["title"][0]["plain_text"]}')
        else:
            print('[INFO] No existing H5N1 database found')
            print('[TIP] Create manually at notion.so, then use database ID')
    except Exception as e:
        print(f'[INFO] Database lookup: {str(e)[:100]}...')

## 2. Update Pipeline Progress

In [ ]:
import json

# Load all pipeline results
print('[INFO] Loading pipeline results...')

stages = {}

# Pre-Stage
stages['Pre-Stage'] = {
    'status': 'Completed',
    'output': '2 epitopes (A:158, A:159), 1 hotspot (A:159, BSA=57.85A²)',
    'date': '2026-09-11'
}

# Stage 1
stages['Stage 1'] = {
    'status': 'Completed',
    'output': '30 RFDiffusion backbones → 20 ProteinMPNN sequences',
    'date': '2026-09-11'
}

# Stage 2
stages['Stage 2'] = {
    'status': 'Completed',
    'output': '30/30 structures passed validation (pLDDT≥80, RMSD<2.0)',
    'date': '2026-09-11'
}

# Stage 3
stages['Stage 3'] = {
    'status': 'Completed',
    'output': 'Top 5 candidates selected (composite scoring: 50% pLDDT + 50% ΔG)',
    'date': '2026-09-11'
}

print('[OK] Pipeline stages:')
for stage, info in stages.items():
    print(f'  {stage}: {info["status"]}')

## 3. Load Top 5 Candidates

In [ ]:
# Load composite scores
df_scores = pd.read_csv('stage3_evaluation/composite_scores.csv')
top5 = df_scores.head(5)

print('[INFO] Top 5 Candidates:')
print(top5[['rank', 'candidate_id', 'plddt', 'delta_g', 'composite_score']].to_string(index=False))

# Create Notion-compatible format
candidates_data = []
for _, row in top5.iterrows():
    candidates_data.append({
        'Rank': int(row['rank']),
        'Candidate ID': row['candidate_id'],
        'pLDDT': round(row['plddt'], 2),
        'ΔG (kcal/mol)': round(row['delta_g'], 2),
        'Composite Score': round(row['composite_score'], 4),
        'Status': 'Ready for Validation'
    })

print(f'\n[OK] Prepared {len(candidates_data)} candidates for Notion')

## 4. Manual Notion Update Instructions

In [ ]:
print('='*70)
print('NOTION DATABASE SETUP INSTRUCTIONS')
print('='*70)

print('''
1. Go to notion.so
2. Create new Database: "H5N1 Antibody Design Pipeline"

3. Create following pages/tables:

   A) Pipeline Progress
      - Stage (text): Pre-Stage, Stage 1, Stage 2, Stage 3
      - Status (select): Completed
      - Output (text): [see below]
      - Date (date): 2026-09-11

   B) Top 5 Candidates
      - Rank (number): 1-5
      - Candidate ID (text): candidate_XXX
      - pLDDT (number): 80-95
      - ΔG (number): -10 to -6
      - Composite Score (number): 0.8+
      - Status (select): Ready for Validation

4. Copy this data into Notion tables:
''')

print('\n--- PIPELINE PROGRESS ---')
for stage, info in stages.items():
    print(f"Stage: {stage}")
    print(f"  Status: {info['status']}")
    print(f"  Output: {info['output']}")
    print(f"  Date: {info['date']}")
    print()

print('--- TOP 5 CANDIDATES ---')
for c in candidates_data:
    print(f"Rank {c['Rank']}: {c['Candidate ID']}")
    print(f"  pLDDT: {c['pLDDT']} | ΔG: {c['ΔG (kcal/mol)']} | Score: {c['Composite Score']}")
    print()

print('='*70)
print('[TIP] Alternative: Use Notion API (set NOTION_TOKEN in Secrets)')
print('='*70)

## 5. Export Data for Notion (JSON/CSV)

In [ ]:
# Save as CSV for easy import
df_notion = pd.DataFrame(candidates_data)
df_notion.to_csv('notion_top5_candidates.csv', index=False)

# Save pipeline stages
df_stages = pd.DataFrame([
    {'Stage': k, 'Status': v['status'], 'Output': v['output'], 'Date': v['date']}
    for k, v in stages.items()
])
df_stages.to_csv('notion_pipeline_progress.csv', index=False)

print('[OK] Export files created:')
print('  - notion_top5_candidates.csv')
print('  - notion_pipeline_progress.csv')
print('\n[TIP] Download and import to Notion Tables')

## 6. Save to GitHub

In [ ]:
if GITHUB_TOKEN and GITHUB_USER:
    !git config --global user.email "dajeong6107@gmail.com"
    !git config --global user.name "Dajeong"
    !git add notion_*.csv
    !git commit -m "Add Notion integration exports" 2>&1 | head -5
    !git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/h5n1-antibody-design.git main 2>&1 | tail -2
    print('[OK] Notion exports pushed to GitHub')
else:
    print('[WARN] No GitHub credentials')

## DONE: Notion Integration Ready

✅ Two options:
1. **Quick**: Import notion_*.csv files to Notion manually
2. **Advanced**: Enable NOTION_TOKEN for API automation